In [1]:
from resonance.api import Beamline
bl = await Beamline.create()

## Beamline Connection Notes

- **`Beamline._conn`** is an instance of the `BCSServer` class, which is part of the autogenerated BCS API package.
- The BCS API is accessed through a lightweight wrapper package installed at `C:\bcs-install`.  
  This wrapper simply points to the original BCS API directory located at:  
  `C:\Beamline Controls\11.0.1 Reflectometer\General Control\Remote Access\BCS API Clients\Python`
- **Note:** The Python virtual environment (`.venv`) is managed by `uv` and may be recreated or overwritten if changes are made.

In [2]:
await bl._conn.start_instrument_acquire("Axis Photonique", acq_time_s=.002)

{'success': True,
 'error description': 'no error',
 'log?': True,
 'elapsed_s': 0.206150099999832,
 'API_delta_t': 0.21213126182556152}

In [3]:
from resonance.api.core.det import get_acquired2d_string
# I just built a dependency injection function that takes in the server, and runs the bcs_request function
# given the locals of the get_acquired2d_string function.

img_str = await get_acquired2d_string(bl._conn, "Axis Photonique")

TypeError: Object of type BCSServer is not JSON serializable

## Acquisition Status & Diagnostic Notes

- **Acquisition via `acquire`**: Working as expected.
- **Issue:**  
  The new `get_acquired2d_string` function fails inside `bcs_request` at the socket response parsing stage.  
  It appears LabVIEW may not recognize `"GetInstrumentAcquired2DString"` as a valid request. 
  Here is the trace back. 
```python
TypeError                                 Traceback (most recent call last)
Cell In[3], line 5
      1 from resonance.api.core.det import get_acquired2d_string
      2 # I just built a dependency injection function that takes in the server, and runs the bcs_request function
      3 # given the locals of the get_acquired2d_string function.
----> 5 img_str = await get_acquired2d_string(bl._conn, "Axis Photonique")

File ~\auto-reflect\src\resonance\api\core\det.py:19, in get_acquired2d_string(conn, name)
     15 async def get_acquired2d_string(conn: BCSz.BCSServer, name: str) -> dict[str, Any]:
     16     """
     17     Get the acquired 2D string from the detector.
     18     """
---> 19     return await conn.bcs_request("GetInstrumentAcquired2DString", dict(locals()))

File c:\Users\Admin\auto-reflect\.venv\Lib\site-packages\bcs\BCSz.py:132, in BCSServer.bcs_request(self, command_name, param_dict, debugging)
    130 if 'self' in param_dict:
    131     del param_dict['self']
--> 132 await self._zmq_socket.send(json.dumps(param_dict).encode())
    133 response_dict = json.loads(await self._zmq_socket.recv())
    134 response_dict['API_delta_t'] = time.time() - api_call_start
...
TypeError: Object of type BCSServer is not JSON serializable
```

### Recent Changes (Potential Root Causes)

1. **LabVIEW has been restarted multiple times.**
2. **Cursor was closed and reopened in advance of users.**  
   – Unsure if all changes were properly saved.

### Troubleshooting Thoughts

- The problem could be at the LabVIEW layer.
- Alternatively, review if any reconfiguration in `BCSz.py` is needed due to the restarts.

> **Next Steps:**  
> - Verify LabVIEW API configuration and request handling for `"GetInstrumentAcquired2DString"`.
> - Check for any unsaved changes or lost state in the Python `BCSz.py` layer after environment restarts.

In [ ]:
import time
import json



# I pulled out the core of the bcs_request function to see what happens at the socket level. 
await bl._conn._zmq_socket.send(
    json.dumps({"command": "GetInstrumentAcquired2DString", "_unused": "_unused", "name": "Axis Photonique"}).encode()
)
# The response seams to be empty. But Idk what type or what the error is since it ist just passed straight into the 
# json decoder in the usual bcs_request function
# >>> await self._zmq_socket.send(json.dumps(param_dict).encode())
# >>> response_dict = json.loads(await self._zmq_socket.recv())
# >>> response_dict['API_delta_t'] = time.time() - api_call_start

res = await bl._conn._zmq_socket.recv()
print(res)
